In [ ]:
import torch\n
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
base_model_path = "Qwen/Qwen3-1.7B"\n
ft_model_path = "../outputs/dpo"\n
question = "A shop sold 12 pens on Monday and 18 pens on Tuesday. How many pens in total?"

In [ ]:
def infer(model_path, prompt):\n
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)\n
    model = AutoModelForCausalLM.from_pretrained(\n
        model_path,\n
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,\n
        device_map="auto",\n
        trust_remote_code=True,\n
    )\n
    text = f"请解答并给出最终答案：\n{prompt}\n答案："\n
    inputs = tokenizer(text, return_tensors="pt").to(model.device)\n
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)\n
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
print("=== Base Model ===")\n
print(infer(base_model_path, question))\n
print("\n=== Fine-tuned Model ===")\n
print(infer(ft_model_path, question))